# HW4 - Design of Experiments

## Problem Definition

A router is used to cut locating notches on printed circuit boards, and the vibration at the board surface is a major source of dimensional variation in the notches. This study investigates whether vibration level is affected by bit size (A: 1/16 in vs. 1/8 in) and cutting speed (B: 40 rpm vs. 90 rpm), whether the two factors interact, and which combination should be used for routine operation to minimize vibration. 

A 2² factorial experiment with four replicates per condition was run, and the response is the resultant vibration measured by three accelerometers (x, y, and z).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import doe

In [ ]:
df = pd.read_csv("circuit-board.csv")
responses = df.to_numpy()

## Main and Interaction Effects

In [ ]:
effect_values = doe.effects(responses, 4)

In [ ]:
doe.plot_effects(responses, 4);

In [ ]:
doe.plot_tornado(responses, 4);

### Discussion

The estimated main effects are 16.64 for bit size (A) and 7.54 for cutting speed (B), and the A×B interaction effect is 8.71. Bit size therefore has the largest effect on vibration, followed by the interaction and then cutting speed. Because both main effects are positive, increasing either factor from its low to its high level raises the average vibration level. The positive interaction effect indicates that the two factors reinforce each other, i.e., the effect of bit size is larger at the high cutting speed than at the low cutting speed. This agrees with the effects plot, where the two lines are not parallel (the B+ line is steeper), suggesting that bit size and cutting speed interact. Since the interaction effect is comparable in magnitude to the main effects, the effect of one factor should not be interpreted independently of the other. These effects are only point estimates, so whether they are statistically significant is determined by the ANOVA below.


## ANOVA

In [ ]:
n = responses.shape[1]
ss_eff = n * effect_values**2
residuals = responses - responses.mean(axis=1, keepdims=True)
sse = (residuals**2).sum()
df_e = 4 * (n - 1)
mse = sse / df_e

F = ss_eff / mse
p = stats.f.sf(F, 1, df_e)

anova_table = pd.DataFrame(
    {
        "SS": [*ss_eff, sse, ss_eff.sum() + sse],
        "df": [1, 1, 1, df_e, responses.size - 1],
        "MS": [*ss_eff, mse, float("nan")],
        "F": [*F, float("nan"), float("nan")],
        "p-value": [*p, float("nan"), float("nan")],
    },
    index=["A", "B", "AB", "Error", "Total"],
)
print(anova_table.to_string(na_rep="", float_format=lambda x: f"{x:.4g}"))


### Discussion

Using a significance level of α = 0.05, all three terms are statistically significant: bit size A (F = 185.3, p = 1.2×10⁻⁸), cutting speed B (F = 38.0, p = 4.8×10⁻⁵), and the AB interaction (F = 50.8, p = 1.2×10⁻⁵). Bit size accounts for 64.8% of the total variation, the interaction for 17.8%, cutting speed for 13.3%, and random error for only 4.2%, so almost all of the variability in vibration is explained by the two factors and their interaction. 


## Recommendation

In [ ]:
mean_responses = responses.mean(axis=1)

mean_table = pd.DataFrame(
    mean_responses.reshape(2, 2).T,
    index=["A-: 1/16 in", "A+: 1/8 in"],
    columns=["B-: 40 rpm", "B+: 90 rpm"],
)
print(mean_table.round(1).to_string())


Because vibration is a major source of dimensional variation in the notches, the goal is to minimize the mean vibration level. Since the A×B interaction is significant, the two factors must be chosen jointly, so we compare the mean value of the four conditions shown above. The larger bit should be avoided, since it increases vibration at both speeds, and the combination of the 1/8 inch bit with 90 rpm is the worst, significantly higher than other condition. With the 1/16 inch bit, vibration is low at both speeds, and the difference between 40 and 90 rpm is not significant. We therefore recommend the 1/16 inch bit for routine operation. Because the mean values corresponing to two cutting speeds have no significant difference on vibration with this bit, it can be chosen on practical grounds: 90 rpm gives the lowest mean (14.9), and a shorter cutting time, whereas 40 rpm is the more conservative choice if the bit is more prone to fracture at higher speed. In either case, the small bit makes the process insensitive to cutting speed, which is desirable for routine operation.

[1] Code was debugged using Claude Sonnet 5.5 (via Claude Code), Nov 14-16, 2026, Anthropic, https://claude.com/claude-code. The same tool was used to polish the language.